In [ ]:
import pickle
import os
import itertools
import scipy
import csv
import pandas as pd
import numpy as np
import scanpy as sc
import scvelo as scv
import seaborn as sns
import gseapy as gp
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.pyplot import rc_context
from matplotlib.colors import LogNorm, Normalize
from scipy import sparse
from scipy import stats
from copy import copy
from scipy.stats import norm
from sklearn.neighbors import KernelDensity
from typing import Tuple
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, Normalize

# Load the data and helpful functions

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.settings.verbosity = 0

In [ ]:
all_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science', 'E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
dbit_datasets = ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
ref_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science']
timepoints = ['E11.5', 'E12.5', 'E13.5']
cellstates = ['MesenchymeProgenitor', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'Precartilage', 'Pericyte',
              'Mesothelium', 'Epithelial(Distal)', 'Epithelial(Proximal)', 'VascularEndothelial',
              'Immune', 'Blood', 'Neuron', 'Heart']
epi_subcluster_cellstates = ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal']
mes_subcluster_cellstates = ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'MatrixMesenchymeProgenitor', 'Precartilage', 'Pericyte']

# figure out color scheme
clusterColors = {'Neuron': 'limeGreen',
                 'MatrixMesenchyme': 'DarkRed',
                 'MesenchymeProgenitor': 'RoyalBlue',
                 'Epithelial(Distal)': 'SkyBlue',
                 'Epithelial(Proximal)': 'Red',
                 'DbitEpithelialBridge': 'Cyan',
                 'SmoothMuscleProgenitor': 'Magenta',
                 'SmoothMuscle': 'Purple',
                 'VascularEndothelial': 'gold',
                 'Mesothelium': 'darkgreen',
                 'Precartilage': 'darkgoldenrod',
                 'Immune': 'DarkOrange',
                 'Heart': 'Gray',
                 'Blood': 'tan',
                 'Pericyte': 'lightpink'}

epiSubclusterColors = {'Proximal': 'Red',
                    'ProximalIntermediate':'darkred',
                    'DistalIntermediate':'darkblue',
                    'Distal':'SkyBlue'}

mesSubclusterColors = {'MP1': 'RoyalBlue',
                       'MP2':'lawngreen',
                       'MP3':'Orange',
                       'MP4':'salmon',
                       'MatrixMesenchyme':'DarkRed',
                       'MatrixMesenchymeProgenitor': 'Red',
                       'SmoothMuscleProgenitor': 'Magenta',
                       'SmoothMuscle':'Purple',
                       'Pericyte': 'lightpink',
                       'Precartilage':'darkgoldenrod'}

phaseDict = {'G1': 'Blue', 'G2M': 'Orange', 'S': 'Green'}

greens = copy(mpl.cm.Greens)
greens.set_under("lightgray")
reds = copy(mpl.cm.Reds(np.linspace(0,1,40)))
reds = mpl.colors.ListedColormap(reds[5:])
reds.set_under("lightgray")

# Add silver as the lower color for the expression colormap
colors = ["silver", "red"]  # Colors for the gradient
expr_cmap = LinearSegmentedColormap.from_list("silver_to_red", colors)
expr_cmap.set_over("red") 
norm = Normalize(vmin=0, vmax=10)

In [ ]:
def plot_spatial_groups(adata, 
                        meta_col,
                        background_adata = False, 
                        palette = False, 
                        show = True, 
                        output_loc = False, 
                        name = 'spatial_plot', 
                        size = 20):
    '''
    input
        - adata: An anndata object. Assumes 14 dbit datasets (split by 'orig.ident'), labeled as dbit in the 'tech' column. Assumes there is an x_loc and y_loc col
        - background_adata: A second anndata object that contains all the cells to plot in the background (if adata is a subset).
            if not provided then it will take adata as the background adata.
        - meta_col: A column in the .obs table or gene in the .var table
        - palette: A dictionary that contains a desired color for each value in the meta_col column or a cmap for a quantitative meta_col.
        - show: True or False on whether to show the plot. Default is False
        - output_loc: Whether or not to save the plot and what folder location. Default is False. 
        - name: The name to give the output pdf if saving the plot
        - size: The dot size for the scatter plot
    returns
        creates spatial scatter plots for the 14 lung spatial datasets. Options to save and show the plot
    '''
    # check if the output_loc folder exists, return an error if not
    if output_loc != False:
        if not os.path.exists(output_loc):
            print('Error: output_loc folder does not exist')
            return
        
    # check if meta_col exists, returns an error if not
    if meta_col not in adata.obs.columns and meta_col not in adata.var.index:
        print('Error: meta_col not in .obs or .var tables')
        return
    
    if type(background_adata) == type(False):
        background_adata = adata
    
    # Make a temp object to work with
    temp_adata = adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in temp_adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []

            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            temp_ax.scatter(x, y, color = 'lightgray', s = size, alpha = 0.3)
        
            # plot the quantitative variable
            temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp)].index
            
            if type(temp_adata.obs[meta_col][0]) != str:
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                if palette != False:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = 'viridis')
            else:
                labels = ['background']
                state_list = temp_adata.obs[meta_col].unique().tolist()
                for state in state_list:
                    temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp) & (temp_adata.obs[meta_col] == state)].index

                    x = [0]
                    y = [0]
                    s = [size]
                    for i in temp_locs:
                        y.append(temp_adata.obs.loc[i, 'y_loc'])
                        x.append(temp_adata.obs.loc[i, 'x_loc'])
                        s.append(size)
                    labels.append(state)

                    if palette != False:
                        temp_ax.scatter(x, y, s = size, color = palette[state])
                    else:
                        temp_ax.scatter(x, y, s = size)
                    if row_count == 0 and col_count == 5:
                        temp_ax.legend(labels = labels, bbox_to_anchor = (1.02, 1))
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)

        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    if output_loc != False:
        plt.savefig(output_loc + name, dpi = 300)
    if show:
        plt.show()
    plt.close()

In [ ]:
# Convert ensambl IDs back into gene names

# Build dictionary for converting GeneID to Gene names
file_path_gene = '/Genomics/chanlab/pengfei/dbit/Manuscript/lungmapping/computation manual/RNAvelocity/Ensembl_geneID_conversion_v2.txt'

# Create an empty dictionary to store the data
data_dict_gene = {}

# Open the file and read its contents
with open(file_path_gene, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        key, value = line.strip().split(',')
        
        # Add key-value pairs to the dictionary
        data_dict_gene[key] = value
flipped_gene_dict = {value: key for key, value in data_dict_gene.items()}

# Build dictionary for converting GeneID to Gene names using the v3 table
file_path_gene_v3 = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/Ensembl_geneID_conversion_v3.txt'

# Create an empty dictionary to store the data
data_dict_gene_v3 = {}

# Open the file and read its contents
with open(file_path_gene_v3, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        value, key = line.strip().split('\t')
        
        # Add key-value pairs to the dictionary
        data_dict_gene_v3[key] = value
flipped_gene_dict_v3 = {value: key for key, value in data_dict_gene.items()}

In [ ]:
# Read in the data
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

In [ ]:
highly_expressed = sc.pp.highly_variable_genes(lung_adata, inplace = False, n_top_genes = 3000)
highly_expressed.index = lung_adata.var.index
top_genes = highly_expressed[highly_expressed['highly_variable'] == True].index
top_genes

# Investigate the subclustered epithelial cell trajectory

In [ ]:
# Look at epithelial trajectory
epi_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_epithelial_subcluster_pseudotime.h5ad')

In [ ]:
print('Cluster, All, DBIT, scRNA')
for i in epi_adata.obs['subcluster_cellstate'].unique():
    print(i, 
          len(epi_adata.obs[(epi_adata.obs['subcluster_cellstate'] == i) ]),
          len(epi_adata.obs[(epi_adata.obs['subcluster_cellstate'] == i) & (epi_adata.obs['tech'] == 'DBiT_seq')]),
          len(epi_adata.obs[(epi_adata.obs['subcluster_cellstate'] == i) & (epi_adata.obs['tech'] == 'scRNA_seq')]))

### look at how cells distribute over the pseudotime

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/epithelial_subcluster/epithelial_subcluster_pseudotime_distributions_per_dataset.pdf')
for exp in all_datasets:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = epi_adata.obs[epi_adata.obs['orig.ident'] == exp]['subcluster_cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = epi_adata.obs[(epi_adata.obs['orig.ident'] == exp) & (epi_adata.obs['subcluster_cellstate'] == state)]['epithelial_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = subclusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 40)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(exp))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = epi_adata.obs[(epi_adata.obs['orig.ident'] == exp) & (epi_adata.obs['subcluster_cellstate'] == state)]['epithelial_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [subclusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/epithelial_subcluster/epithelial_subcluster_pseudotime_distributions_per_timepoint.pdf')
for timepoint in timepoints:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = epi_adata.obs[epi_adata.obs['timepoint'] == timepoint]['subcluster_cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = epi_adata.obs[(epi_adata.obs['timepoint'] == timepoint) & (epi_adata.obs['subcluster_cellstate'] == state)]['epithelial_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = subclusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 40)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(timepoint))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = epi_adata.obs[(epi_adata.obs['timepoint'] == timepoint) & (epi_adata.obs['subcluster_cellstate'] == state)]['epithelial_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [subclusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

### Look at modules

In [ ]:
# Use the scRNA only modules
epi_sub_modules = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/epithelial_subcluster/Lung_epithelial_subcluster_scRNA_only_monocle_gene_modules_genes_above_0.1morans.txt', index_col = 0)

epi_sub_modules['ensambl_id'] = epi_sub_modules['id']
for i in epi_sub_modules.index:
    if epi_sub_modules.loc[i, 'ensambl_id'] in data_dict_gene_v3.keys():
        epi_sub_modules.loc[i, 'id'] = data_dict_gene_v3[epi_sub_modules.loc[i, 'ensambl_id']]

In [ ]:
modules = epi_sub_modules['module'].unique()
modules.sort()
module_list = ['module_{}_score'.format(i) for i in modules]

# use the scanpy score genes function to calculate the module score of each module for each cell
for temp_module in modules:
    gene_ids = epi_sub_modules[epi_sub_modules['module'] == temp_module]['id']
    sc.tl.score_genes(epi_adata, gene_ids, ctrl_size = len(gene_ids), score_name = 'module_{}_score'.format(temp_module), use_raw = False)
    
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(epi_adata, color = module_list, save = '_epi_subcluster_module_scores.pdf', show = False)

module_scores_df = pd.DataFrame(index = epi_adata.obs.index, columns = modules)

for i in module_scores_df.index:
    for j in module_scores_df.columns:
        module_scores_df.loc[i, j] = epi_adata.obs.loc[i, 'module_{}_score'.format(j)]
        
module_scores_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/epithelial_subcluster/epithelial_subcluster_module_scores.csv')

### Plot marker genes over the pseudotime
- Plot using only scRNA cells

In [ ]:
itr_ranges = []
for i in np.arange(2, max(epi_adata.obs['epithelial_pseudotime']), 2):
    itr_ranges.append('{}_{}'.format(i-2, i))

# Calculate the avg expression of each of the 1 step bins
avg_itr_expr_df = pd.DataFrame(columns = epi_adata.var.index, index = itr_ranges)

for i in avg_itr_expr_df.index:
    lower = float(i.split('_')[0])
    upper = float(i.split('_')[1])
    cells = epi_adata.obs[(epi_adata.obs['tech'] == 'scRNA_seq') & (epi_adata.obs['epithelial_pseudotime'] >= lower) & (epi_adata.obs['epithelial_pseudotime'] < upper)].index
    temp = epi_adata[epi_adata.obs.index.isin(cells)].copy()
    
    expr = temp.X.todense()

    avg_itr_expr_df.loc[i] = temp.X.todense().mean(axis = 0)
avg_itr_expr_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/epithelial_subcluster/epithelial_subcluster_avg_itr_2step.txt', sep = '\t')

In [ ]:
# calculate the zscores of genes
z_expr_df = avg_itr_expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

# Calculate the Z score
mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)
z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

### Use linregress to find genes with a high slope in the middle region (10 - 29)

In [ ]:
# Look at only highly variable genes in scRNA only cells
highly_expressed = sc.pp.highly_variable_genes(epi_adata[epi_adata.obs['tech'] != 'DBiT_seq'], inplace = False, n_top_genes = 2000)
highly_expressed.index = epi_adata.var.index
top_genes_scRNA = highly_expressed[highly_expressed['highly_variable'] == True].index
top_genes_scRNA

In [ ]:
# For each gene, calculate the regression and save the values to a txt file
genes = top_genes_scRNA

epi_file = open('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/epithelial_subcluster/epithelial_subcluster_scRNA_only_linregress.txt', 'w')
epi_file.write('Gene' + '\t' + 'slope' + '\t' + 'intercept' + '\t' + 'r_val' + '\t' + 'p_val' + '\t' + 'stderr' + '\n')
epi_file.write('\n')

for gene in genes:
    temp_expr = epi_adata[(epi_adata.obs['tech'] == 'scRNA_seq')][:, gene].X.todense().flatten()
    x = epi_adata.obs[(epi_adata.obs['tech'] == 'scRNA_seq')]['epithelial_pseudotime']
    
    slope, intercept, r, p, se = stats.linregress(x, temp_expr)
    
    epi_file.write(gene + '\t')
    epi_file.write('{}\t{}\t{}\t{}\t{}'.format(slope, intercept, r, p, se))
    epi_file.write('\n')

epi_file.close()

In [ ]:
test = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/epithelial_subcluster/epithelial_subcluster_scRNA_only_linregress.txt', sep = '\t')
top_slope_genes = test.sort_values('slope', ascending = False)['Gene'][0:30].tolist()
bot_slope_genes = test.sort_values('slope', ascending = True)['Gene'][0:30].tolist()

# Look at the SmoothMuscle Trajectory in the subclustered datasets

In [ ]:
# Look at epithelial trajectory
mes_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_mesenchyme_subcluster_pseudotime.h5ad')

In [ ]:
# Subset to only the cells in this pseudotime
# I added a subset of 25 because there are some outlier MP1 cells that were located over near MP2 and got a much different pseudotime value
sm_sub = mes_adata[(mes_adata.obs['subcluster_cellstate'].isin(['MP1', 'SmoothMuscleProgenitor', 'SmoothMuscle']))].copy()

### Look at how cells distribute over the pseudotime

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle_subcluster/smooth_muscle_subcluster_pseudotime_distributions_per_dataset.pdf')
for exp in all_datasets:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = sm_sub.obs[sm_sub.obs['orig.ident'] == exp]['subcluster_cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = sm_sub.obs[(sm_sub.obs['orig.ident'] == exp) & (sm_sub.obs['subcluster_cellstate'] == state)]['smooth_muscle_subcluster_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = mesSubclusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 40)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(exp))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = sm_sub.obs[(sm_sub.obs['orig.ident'] == exp) & (sm_sub.obs['subcluster_cellstate'] == state)]['smooth_muscle_subcluster_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [mesSubclusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle_subcluster/smooth_muscle_subcluster_pseudotime_distributions_per_timepoint.pdf')
for timepoint in timepoints:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = sm_sub.obs[sm_sub.obs['timepoint'] == timepoint]['subcluster_cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = sm_sub.obs[(sm_sub.obs['timepoint'] == timepoint) & (sm_sub.obs['subcluster_cellstate'] == state)]['smooth_muscle_subcluster_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = mesSubclusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 40)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(timepoint))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = sm_sub.obs[(sm_sub.obs['timepoint'] == timepoint) & (sm_sub.obs['subcluster_cellstate'] == state)]['smooth_muscle_subcluster_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [mesSubclusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

### Look at gene modules

In [ ]:
# Use the scRNA only modules
sm_sub_modules = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/mesenchyme_subcluster/Lung_mesenchyme_subcluster_monocle_gene_modules_genes_above_0.1morans.txt', index_col = 0)

sm_sub_modules['ensambl_id'] = sm_sub_modules['id']
for i in sm_sub_modules.index:
    if sm_sub_modules.loc[i, 'ensambl_id'] in data_dict_gene_v3.keys():
        sm_sub_modules.loc[i, 'id'] = data_dict_gene_v3[sm_sub_modules.loc[i, 'ensambl_id']]

In [ ]:
modules = sm_sub_modules['module'].unique()
modules.sort()
module_list = ['module_{}_score'.format(i) for i in modules]

# use the scanpy score genes function to calculate the module score of each module for each cell
for temp_module in modules:
    gene_ids = sm_sub_modules[sm_sub_modules['module'] == temp_module]['id']
    sc.tl.score_genes(sm_sub, gene_ids, ctrl_size = len(gene_ids), score_name = 'module_{}_score'.format(temp_module), use_raw = False)
    
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(sm_sub, color = module_list, save = '_sm_subcluster_module_scores.pdf', show = False)

module_scores_df = pd.DataFrame(index = sm_sub.obs.index, columns = modules)

for i in module_scores_df.index:
    for j in module_scores_df.columns:
        module_scores_df.loc[i, j] = sm_sub.obs.loc[i, 'module_{}_score'.format(j)]
        
module_scores_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle_subcluster/smooth_muscle_subcluster_module_scores.csv')

### Plot marker genes over the pseudotime
- Using scRNA only

In [ ]:
itr_ranges = []
for i in np.arange(1, max(sm_sub.obs['smooth_muscle_subcluster_pseudotime']), 1):
    itr_ranges.append('{}_{}'.format(i-1, i))

# Calculate the avg expression of each of the 1 step bins
avg_itr_expr_df = pd.DataFrame(columns = sm_sub.var.index, index = itr_ranges)

for i in avg_itr_expr_df.index:
    lower = float(i.split('_')[0])
    upper = float(i.split('_')[1])
    cells = sm_sub.obs[(sm_sub.obs['tech'] == 'scRNA_seq') & (sm_sub.obs['smooth_muscle_subcluster_pseudotime'] >= lower) & (sm_sub.obs['smooth_muscle_subcluster_pseudotime'] < upper)].index
    temp = sm_sub[sm_sub.obs.index.isin(cells)].copy()
    
    expr = temp.X.todense()

    avg_itr_expr_df.loc[i] = temp.X.todense().mean(axis = 0)
avg_itr_expr_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle_subcluster/smooth_muscle_subcluster_avg_itr_1step.txt', sep = '\t')

In [ ]:
# calculate the zscores of genes
z_expr_df = avg_itr_expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

# Calculate the Z score
mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)
z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

### Run linregress to find genes that have drastic slopes over the pseudotime

In [ ]:
# Look at only highly variable genes in scRNA only cells
highly_expressed = sc.pp.highly_variable_genes(sm_sub[sm_sub.obs['tech'] != 'DBiT_seq'], inplace = False, n_top_genes = 2000)
highly_expressed.index = sm_sub.var.index
top_genes_scRNA = highly_expressed[highly_expressed['highly_variable'] == True].index
top_genes_scRNA

In [ ]:
# For each gene, calculate the regression and save the values to a txt file
genes = top_genes_scRNA

sm_sub_file = open('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle_subcluster/smooth_muscle_subcluster_scRNA_only_linregress.txt', 'w')
sm_sub_file.write('Gene' + '\t' + 'slope' + '\t' + 'intercept' + '\t' + 'r_val' + '\t' + 'p_val' + '\t' + 'stderr' + '\n')
sm_sub_file.write('\n')

for gene in genes:
    temp_expr = sm_sub[(sm_sub.obs['tech'] == 'scRNA_seq')][:, gene].X.todense().flatten()
    x = sm_sub.obs[(sm_sub.obs['tech'] == 'scRNA_seq')]['mesenchyme_subcluster_pseudotime']
    
    slope, intercept, r, p, se = stats.linregress(x, temp_expr)
    
    sm_sub_file.write(gene + '\t')
    sm_sub_file.write('{}\t{}\t{}\t{}\t{}'.format(slope, intercept, r, p, se))
    sm_sub_file.write('\n')

sm_sub_file.close()

In [ ]:
test = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle_subcluster/smooth_muscle_subcluster_scRNA_only_linregress.txt', sep = '\t')
top_slope_genes = test.sort_values('slope', ascending = False)['Gene'][0:30].tolist()
bot_slope_genes = test.sort_values('slope', ascending = True)['Gene'][0:30].tolist()

# Look at the Matrix Mesenchyme trajectory

In [ ]:
# Subset to only the cells in this pseudotime
syn_sub = mes_adata[mes_adata.obs['synthetic_mesenchyme_subcluster_pseudotime'] > -100].copy()

### Look at how the cells distribute over the pseudotime

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme_subcluster/synthetic_mesenchyme_subcluster_pseudotime_distributions_per_dataset.pdf')
for exp in all_datasets:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = syn_sub.obs[syn_sub.obs['orig.ident'] == exp]['subcluster_cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = syn_sub.obs[(syn_sub.obs['orig.ident'] == exp) & (syn_sub.obs['subcluster_cellstate'] == state)]['synthetic_mesenchyme_subcluster_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = mesSubclusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 40)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(exp))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = syn_sub.obs[(syn_sub.obs['orig.ident'] == exp) & (syn_sub.obs['subcluster_cellstate'] == state)]['synthetic_mesenchyme_subcluster_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [mesSubclusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme_subcluster/synthetic_mesenchyme_subcluster_pseudotime_distributions_per_timepoint.pdf')
for timepoint in timepoints:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = syn_sub.obs[syn_sub.obs['timepoint'] == timepoint]['subcluster_cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = syn_sub.obs[(syn_sub.obs['timepoint'] == timepoint) & (syn_sub.obs['subcluster_cellstate'] == state)]['synthetic_mesenchyme_subcluster_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = mesSubclusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 40)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(timepoint))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = syn_sub.obs[(sm_sub.obs['timepoint'] == timepoint) & (syn_sub.obs['subcluster_cellstate'] == state)]['synthetic_mesenchyme_subcluster_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [mesSubclusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

### Look at the modules

In [ ]:
# Use the scRNA only modules
syn_sub_modules = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/synthetic_mesenchyme_subcluster/Lung_synthetic_mesenchyme_subcluster_monocle_gene_modules_genes_above_0.1morans.txt', index_col = 0)

syn_sub_modules['ensambl_id'] = syn_sub_modules['id']
for i in syn_sub_modules.index:
    if syn_sub_modules.loc[i, 'ensambl_id'] in data_dict_gene_v3.keys():
        syn_sub_modules.loc[i, 'id'] = data_dict_gene_v3[syn_sub_modules.loc[i, 'ensambl_id']]

### Plot expression over the pseudotime

In [ ]:
itr_ranges = []
for i in np.arange(1, max(syn_sub.obs['synthetic_mesenchyme_subcluster_pseudotime']), 1):
    itr_ranges.append('{}_{}'.format(i-1, i))

# Calculate the avg expression of each of the 1 step bins
avg_itr_expr_df = pd.DataFrame(columns = syn_sub.var.index, index = itr_ranges)

for i in avg_itr_expr_df.index:
    lower = float(i.split('_')[0])
    upper = float(i.split('_')[1])
    cells = syn_sub.obs[(syn_sub.obs['tech'] == 'scRNA_seq') & (syn_sub.obs['synthetic_mesenchyme_subcluster_pseudotime'] >= lower) & (syn_sub.obs['synthetic_mesenchyme_subcluster_pseudotime'] < upper)].index
    temp = syn_sub[syn_sub.obs.index.isin(cells)].copy()
    
    expr = temp.X.todense()

    avg_itr_expr_df.loc[i] = temp.X.todense().mean(axis = 0)
avg_itr_expr_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme_subcluster/synthetic_mesenchyme_subcluster_avg_itr_1step.txt', sep = '\t')

In [ ]:
# calculate the zscores of genes
z_expr_df = avg_itr_expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

# Calculate the Z score
mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)
z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

### Use linear regression to identify genes that change in expression

In [ ]:
# Look at only highly variable genes in scRNA only cells
highly_expressed = sc.pp.highly_variable_genes(syn_sub[syn_sub.obs['tech'] != 'DBiT_seq'], inplace = False, n_top_genes = 2000)
highly_expressed.index = syn_sub.var.index
top_genes_scRNA = highly_expressed[highly_expressed['highly_variable'] == True].index
top_genes_scRNA

# For each gene, calculate the regression and save the values to a txt file
genes = top_genes_scRNA

syn_sub_file = open('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme_subcluster/synthetic_mesenchyme_subcluster_scRNA_only_linregress.txt', 'w')
syn_sub_file.write('Gene' + '\t' + 'slope' + '\t' + 'intercept' + '\t' + 'r_val' + '\t' + 'p_val' + '\t' + 'stderr' + '\n')
syn_sub_file.write('\n')

for gene in genes:
    temp_expr = syn_sub[(syn_sub.obs['tech'] == 'scRNA_seq')][:, gene].X.todense().flatten()
    x = syn_sub.obs[(syn_sub.obs['tech'] == 'scRNA_seq')]['synthetic_mesenchyme_subcluster_pseudotime']
    
    slope, intercept, r, p, se = stats.linregress(x, temp_expr)
    
    syn_sub_file.write(gene + '\t')
    syn_sub_file.write('{}\t{}\t{}\t{}\t{}'.format(slope, intercept, r, p, se))
    syn_sub_file.write('\n')

syn_sub_file.close()

In [ ]:
test = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme_subcluster/synthetic_mesenchyme_subcluster_scRNA_only_linregress.txt', sep = '\t')
top_slope_genes = test.sort_values('slope', ascending = False)['Gene'][0:30].tolist()
bot_slope_genes = test.sort_values('slope', ascending = True)['Gene'][0:30].tolist()

In [ ]:
# Look at the mesenchyme 1 trajectory
sm = lung_adata[lung_adata.obs['cellstate'].isin(['MesenchymeProgenitor', 'SmoothMuscle', 'SmoothMuscleProgenitor'])].copy()

### look at how cells distribute over the pseudotime

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle/smooth_muscle_pseudotime_distributions_per_dataset.pdf')
for exp in all_datasets:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = sm.obs[sm.obs['orig.ident'] == exp]['cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = sm.obs[(sm.obs['orig.ident'] == exp) & (sm.obs['cellstate'] == state)]['smooth_muscle_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = clusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 25)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(exp))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = sm.obs[(sm.obs['orig.ident'] == exp) & (sm.obs['cellstate'] == state)]['smooth_muscle_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [clusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle/smooth_muscle_pseudotime_distributions_per_timepoint.pdf')
for timepoint in timepoints:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = sm.obs[sm.obs['timepoint'] == timepoint]['cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = sm.obs[(sm.obs['timepoint'] == timepoint) & (sm.obs['cellstate'] == state)]['smooth_muscle_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = clusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 25)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(timepoint))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = sm.obs[(sm.obs['timepoint'] == timepoint) & (sm.obs['cellstate'] == state)]['smooth_muscle_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [clusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

### Look at smooth muscle modules

In [ ]:
# Use the scRNA only modules
sm_modules = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/smooth_muscle/Lung_smooth_muscle_scRNA_only_integrated_monocle_gene_modules_genes_above_0.1morans.txt', index_col = 0)

In [ ]:
sm_modules['ensambl_id'] = sm_modules['id']
for i in sm_modules.index:
    if sm_modules.loc[i, 'ensambl_id'] in data_dict_gene_v3.keys():
        sm_modules.loc[i, 'id'] = data_dict_gene_v3[sm_modules.loc[i, 'ensambl_id']]

In [ ]:
modules = sm_modules['module'].unique()
modules.sort()
module_list = ['module_{}_score'.format(i) for i in modules]

# use the scanpy score genes function to calculate the module score of each module for each cell
for temp_module in modules:
    gene_ids = sm_modules[sm_modules['module'] == temp_module]['id']
    sc.tl.score_genes(sm, gene_ids, ctrl_size = len(gene_ids), score_name = 'module_{}_score'.format(temp_module), use_raw = False)
    
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(sm, color = module_list, save = '_smooth_muscle_module_scores.pdf', show = False)

module_scores_df = pd.DataFrame(index = sm.obs.index, columns = modules)

for i in module_scores_df.index:
    for j in module_scores_df.columns:
        module_scores_df.loc[i, j] = sm.obs.loc[i, 'module_{}_score'.format(j)]
        
module_scores_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle/smooth_muscle_module_scores.csv')

### Plot a heatmap of the top modules over the pseudotime

In [ ]:
itr_ranges = []
for i in np.arange(1, max(sm.obs['smooth_muscle_pseudotime']), 1):
    itr_ranges.append('{}_{}'.format(i-1, i))

# Calculate the avg expression of each of the 1 step bins
avg_itr_expr_df = pd.DataFrame(columns = sm.var.index, index = itr_ranges)

for i in avg_itr_expr_df.index:
    lower = float(i.split('_')[0])
    upper = float(i.split('_')[1])
    cells = sm.obs[(sm.obs['smooth_muscle_pseudotime'] >= lower) & (sm.obs['smooth_muscle_pseudotime'] < upper)].index
    #print(len(cells))
    temp = sm[sm.obs.index.isin(cells)].copy()
    
    expr = temp.X.todense()

    avg_itr_expr_df.loc[i] = temp.X.todense().mean(axis = 0)
avg_itr_expr_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle/smooth_muscle_avg_itr_1step.txt', sep = '\t')

In [ ]:
# calculate the zscores of genes
z_expr_df = avg_itr_expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

# Calculate the Z score
mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)
z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

### look at how cells distribute over the pseudotime

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme/synthetic_mesenchyme_pseudotime_distributions_per_dataset.pdf')
for exp in all_datasets:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = syn.obs[syn.obs['orig.ident'] == exp]['cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = syn.obs[(syn.obs['orig.ident'] == exp) & (syn.obs['cellstate'] == state)]['synthetic_mesenchyme_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = clusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 20)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(exp))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = syn.obs[(syn.obs['orig.ident'] == exp) & (syn.obs['cellstate'] == state)]['synthetic_mesenchyme_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [clusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

In [ ]:
# Plot the kde's of cells in each datasets, save them to a collective pdf
pp = PdfPages('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme/synthetic_mesenchyme_pseudotime_distributions_per_timepoint.pdf')
for timepoint in timepoints:
    fig, axs = plt.subplots(1, 2, figsize = (12, 3))
    cell_states = syn.obs[syn.obs['timepoint'] == timepoint]['cellstate'].unique().tolist()

    temp_ax = axs[0]
    labels = []
    for state in cell_states:
        temp_values = syn.obs[(syn.obs['timepoint'] == timepoint) & (syn.obs['cellstate'] == state)]['synthetic_mesenchyme_pseudotime']
        if len(temp_values) > 1:
            sns.kdeplot(temp_values, ax = temp_ax, color = clusterColors[state])
            labels.append('{}_n={}'.format(state, len(temp_values)))
    temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
    temp_ax.set_xlim(0, 20)
    #temp_ax.set_ylim(0, 0.45)
    temp_ax.set_title('{}'.format(timepoint))
      
                            
    temp_ax = axs[1]  
    values = np.zeros(len(cell_states))
    count = 0
    for state in cell_states:
        temp_values = syn.obs[(syn.obs['timepoint'] == timepoint) & (syn.obs['cellstate'] == state)]['synthetic_mesenchyme_pseudotime']
        values[count] = len(temp_values)
        count += 1
    temp_ax.pie(values, labels = cell_states, autopct='%1.1f%%', colors = [clusterColors[key] for key in cell_states])
    
    plt.tight_layout()
    #plt.show()
    pp.savefig()
    plt.close()
pp.close()

In [ ]:
# Look at how the proximal cluster distributes over the pseudotime in only DBiT seq cells
fig, axs = plt.subplots(1, 1, figsize = (9, 3))
temp_ax = axs
labels = []
for cellstate in syn.obs['cellstate'].unique():
    temp_values = syn.obs[(syn.obs['tech'] != 'DBiT_seq') & (syn.obs['cellstate'] == cellstate)]['synthetic_mesenchyme_pseudotime']
    if len(temp_values) > 1:
        sns.kdeplot(temp_values, ax = temp_ax)
        labels.append('{}_n={}'.format(cellstate, len(temp_values)))
temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
temp_ax.set_xlim(0, 20)
#temp_ax.set_ylim(0, 0.45)
temp_ax.set_title('Synthetic Mesenchyme')

plt.tight_layout()
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3_synthetic_mesenchyme_pseudotime_density.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at how the proximal cluster distributes over the pseudotime in only DBiT seq cells
fig, axs = plt.subplots(1, 1, figsize = (9, 3))
temp_ax = axs
labels = []
for cellstate in syn.obs['cellstate'].unique():
    temp_values = syn.obs[(syn.obs['cellstate'] == cellstate)]['synthetic_mesenchyme_pseudotime']
    if len(temp_values) > 1:
        sns.kdeplot(temp_values, ax = temp_ax)
        labels.append('{}_n={}'.format(cellstate, len(temp_values)))
temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
temp_ax.set_xlim(0, 20)
#temp_ax.set_ylim(0, 0.45)
temp_ax.set_title('Synthetic Mesenchyme')

plt.tight_layout()
plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3_synthetic_mesenchyme_pseudotime_density.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at how the proximal cluster distributes over the pseudotime in only DBiT seq cells
fig, axs = plt.subplots(1, 1, figsize = (9, 3))
temp_ax = axs
labels = []
for timepoint in timepoints:
    temp_values = syn.obs[(syn.obs['tech'] != 'DBiT_seq') & (syn.obs['timepoint'] == timepoint)]['synthetic_mesenchyme_pseudotime']
    if len(temp_values) > 1:
        sns.kdeplot(temp_values, ax = temp_ax)
        labels.append('{}_n={}'.format(timepoint, len(temp_values)))
temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
temp_ax.set_xlim(0, 20)
#temp_ax.set_ylim(0, 0.45)
temp_ax.set_title('Synthetic Mesenchyme')

plt.tight_layout()
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3_synthetic_mesenchyme_pseudotime_density.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at how the proximal cluster distributes over the pseudotime in only DBiT seq cells
fig, axs = plt.subplots(1, 1, figsize = (9, 3))
temp_ax = axs
labels = []
for timepoint in timepoints:
    temp_values = syn.obs[(syn.obs['tech'] == 'DBiT_seq')& (syn.obs['timepoint'] == timepoint)]['synthetic_mesenchyme_pseudotime']
    if len(temp_values) > 1:
        sns.kdeplot(temp_values, ax = temp_ax)
        labels.append('{}_n={}'.format(timepoint, len(temp_values)))
temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
temp_ax.set_xlim(0, 20)
#temp_ax.set_ylim(0, 0.45)
temp_ax.set_title('Synthetic Mesenchyme (Dbit Only)')

plt.tight_layout()
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3_synthetic_mesenchyme_pseudotime_density.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at how the proximal cluster distributes over the pseudotime in only DBiT seq cells
fig, axs = plt.subplots(1, 1, figsize = (9, 3))
temp_ax = axs
labels = []
for timepoint in timepoints:
    temp_values = syn.obs[(syn.obs['timepoint'] == timepoint) & (syn.obs['cellstate'] == 'SyntheticMesenchyme')]['synthetic_mesenchyme_pseudotime']
    if len(temp_values) > 1:
        sns.kdeplot(temp_values, ax = temp_ax)
        labels.append('{}_n={}'.format(timepoint, len(temp_values)))
temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
temp_ax.set_xlim(0, 20)
#temp_ax.set_ylim(0, 0.45)
temp_ax.set_title('Synthetic Mesenchyme Over Time')

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Look at how the proximal cluster distributes over the pseudotime in only DBiT seq cells
fig, axs = plt.subplots(1, 1, figsize = (9, 3))
temp_ax = axs
labels = []
for timepoint in timepoints:
    temp_values = syn.obs[(syn.obs['tech'] == 'DBiT_seq') & (syn.obs['timepoint'] == timepoint) & (syn.obs['cellstate'] == 'SyntheticMesenchyme')]['synthetic_mesenchyme_pseudotime']
    if len(temp_values) > 1:
        sns.kdeplot(temp_values, ax = temp_ax)
        labels.append('{}_n={}'.format(timepoint, len(temp_values)))
temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
temp_ax.set_xlim(0, 20)
#temp_ax.set_ylim(0, 0.45)
temp_ax.set_title('Synthetic Mesenchyme (DBiT Only)')

plt.tight_layout()
plt.show()
plt.close()

### Look at the modules

In [ ]:
# Use the scRNA only modules
syn_modules = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/synthetic_mesenchyme/Lung_synthetic_mesenchyme_scRNA_only_monocle_gene_modules_genes_above_0.1morans.txt', index_col = 0)

In [ ]:
syn_modules['ensambl_id'] = syn_modules['id']
for i in syn_modules.index:
    if syn_modules.loc[i, 'ensambl_id'] in data_dict_gene_v3.keys():
        syn_modules.loc[i, 'id'] = data_dict_gene_v3[syn_modules.loc[i, 'ensambl_id']]

In [ ]:
modules = syn_modules['module'].unique()
modules.sort()
module_list = ['module_{}_score'.format(i) for i in modules]

# use the scanpy score genes function to calculate the module score of each module for each cell
for temp_module in modules:
    gene_ids = syn_modules[syn_modules['module'] == temp_module]['id']
    sc.tl.score_genes(syn, gene_ids, ctrl_size = len(gene_ids), score_name = 'module_{}_score'.format(temp_module), use_raw = False)
    
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(syn, color = module_list, save = '_synthetic_mesenchyme_module_scores.pdf', show = False)

module_scores_df = pd.DataFrame(index = syn.obs.index, columns = modules)

for i in module_scores_df.index:
    for j in module_scores_df.columns:
        module_scores_df.loc[i, j] = syn.obs.loc[i, 'module_{}_score'.format(j)]
        
module_scores_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme/synthetic_mesenchyme_module_scores.csv')

In [ ]:
syn_modules[syn_modules['module'] == 12]

In [ ]:
syn_modules[syn_modules['id'].isin(['Ptn', 'Col1a1', 'Wnt2', 'Cdk1', 'Prrx1', 'Fgf10', 'Adamts9', 'Tmtc2', 'Rspo2', 'Piezo2', 'Cenpa', 'Cdk4', 'Cenpw',
              'Cenpn', 'Col3a1', 'Col5a2', 'Col5a1', 'Col1a2', 'Erbb4', 'Grid2', 'Lrfn5', 'Pbx1', 'Ccnb1', 'Sparc', 'Bmp3'])]

In [ ]:
for module in module_list:
    plot_spatial_groups(adata = syn,
                       background_adata = lung_adata,
                       meta_col = module,
                       output_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme/modules/',
                       name = 'synthetic_mesenchyme_{}.pdf'.format(module))

In [ ]:
itr_ranges = []
for i in np.arange(1, max(syn.obs['synthetic_mesenchyme_pseudotime']), 1):
    itr_ranges.append('{}_{}'.format(i-1, i))

# Calculate the avg expression of each of the 1 step bins
avg_itr_expr_df = pd.DataFrame(columns = syn.var.index, index = itr_ranges)

for i in avg_itr_expr_df.index:
    lower = float(i.split('_')[0])
    upper = float(i.split('_')[1])
    cells = syn.obs[(syn.obs['synthetic_mesenchyme_pseudotime'] >= lower) & (syn.obs['synthetic_mesenchyme_pseudotime'] < upper)].index
    #print(len(cells))
    temp = syn[syn.obs.index.isin(cells)].copy()
    
    expr = temp.X.todense()

    avg_itr_expr_df.loc[i] = temp.X.todense().mean(axis = 0)
avg_itr_expr_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/synthetic_mesenchyme/synthetic_mesenchyme_avg_itr_1step.txt', sep = '\t')

In [ ]:
# calculate the zscores of genes
z_expr_df = avg_itr_expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

# Calculate the Z score
mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)
z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

In [ ]:
end = reordered_labels[-140:-80]
middle = reordered_labels[-80:]

In [ ]:
genes_of_interest = syn_modules[syn_modules['module'].isin([7, 11, 10, 9, 3, 13, 1, 6])]['id']

good_genes = []
for gene in genes_of_interest:
    if gene in z_expr_df.columns:
        good_genes.append(gene)

use_labels = ['Ptn', 'Col1a1', 'Wnt2', 'Cdk1', 'Prrx1', 'Fgf10', 'Adamts9', 'Tmtc2', 'Rspo2', 'Piezo2', 'Cenpa', 'Cdk4','Sema3a',
              'Col3a1', 'Col5a2', 'Col5a1', 'Col1a2', 'Erbb4', 'Grid2', 'Lrfn5', 'Pbx1', 'Ccnb1', 'Sparc', 'Bmp3', 'Foxp2']
for gene in use_labels:
    if gene not in good_genes:
        good_genes.append(gene)

temp = z_expr_df[good_genes].copy()

vcenter = 0
#vmin, vmax = temp.min().min(), temp.max().max()
vmin, vmax = -2, 3
normalize = mcolors.TwoSlopeNorm(vcenter=vcenter, vmin=vmin, vmax=vmax)
colormap = cm.RdYlBu_r

# create a clustermap to get the clustered order
g = sns.clustermap(temp.T, figsize = (5, 10), col_cluster = False, cmap=cm.RdYlBu_r, vmin = -4, vmax = 4, yticklabels=False)
plt.close()

# reorder the clustermap to look nicer
g.dendrogram_row.reordered_ind.reverse()
reordered_labels = temp.T.index[g.dendrogram_row.reordered_ind].tolist()

# reorder the clusters to capture pseudotime better
start = reordered_labels[:-140]
end = reordered_labels[-140:-80]
middle = reordered_labels[-80:]

reordered_labels = start + middle + end

g = sns.clustermap(temp.T.loc[reordered_labels], figsize = (5, 10), col_cluster = False, row_cluster = False,
                   cmap=cm.RdYlBu_r, vmin = -4, vmax = 4, yticklabels=False)

use_ticks = [reordered_labels.index(label) + .5 for label in use_labels]
g.ax_heatmap.set(yticks=use_ticks, yticklabels=use_labels)
plt.subplots_adjust(bottom = 0.3, right = 0.7)

g.ax_heatmap.set(yticks=use_ticks, yticklabels=use_labels)
plt.subplots_adjust(bottom = 0.3, right = 0.7)
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3g_synthetic_mesenchyme_pseudotime_heatmap.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the mesenchyme 1 trajectory
pc = lung_adata[lung_adata.obs['cellstate'].isin(['MesenchymeProgenitor1', 'MesenchymeProgenitor3', 'Precartilage'])].copy()

In [ ]:
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(pc, color = ['precartilage_pseudotime'], size = 5, wspace = 0.35)

In [ ]:
# Look at how the proximal cluster distributes over the pseudotime in only DBiT seq cells
fig, axs = plt.subplots(1, 1, figsize = (9, 3))
temp_ax = axs
labels = []
for cellstate in pc.obs['cellstate'].unique():
    temp_values = pc.obs[(pc.obs['cellstate'] == cellstate)]['precartilage_pseudotime']
    if len(temp_values) > 1:
        sns.kdeplot(temp_values, ax = temp_ax)
        labels.append('{}_n={}'.format(cellstate, len(temp_values)))
temp_ax.legend(labels, bbox_to_anchor = (1.01, 1.01))
temp_ax.set_xlim(0, 20)
#temp_ax.set_ylim(0, 0.45)
temp_ax.set_title('Precartilage')

plt.tight_layout()
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3_synthetic_mesenchyme_pseudotime_density.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# For each gene, calculate the regression and save the values to a txt file
genes = avg_itr_expr_df.columns

sm_file = open('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle/smooth_muscle_linregress.txt', 'w')
sm_file.write('Gene' + '\t' + 'slope' + '\t' + 'intercept' + '\t' + 'r_val' + '\t' + 'p_val' + '\t' + 'stderr' + '\n')
sm_file.write('\n')

for gene in genes:
    temp_expr = sm[:, gene].X.todense().flatten()
    x = sm.obs['smooth_muscle_pseudotime']
    
    slope, intercept, r, p, se = stats.linregress(x, temp_expr)
    
    sm_file.write(gene + '\t')
    sm_file.write('{}\t{}\t{}\t{}\t{}'.format(slope, intercept, r, p, se))
    sm_file.write('\n')

sm_file.close()

In [ ]:
test = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/pseudotime/smooth_muscle/smooth_muscle_linregress.txt', sep = '\t')
top_slope_genes = test.sort_values('slope', ascending = False)['Gene'][0:10].tolist()
bot_slope_genes = test.sort_values('slope', ascending = True)['Gene'][0:10].tolist()

In [ ]:
module_1_genes = sm_modules[sm_modules['module'] == 1]['id'].tolist()
module_2_genes = sm_modules[sm_modules['module'] == 2]['id'].tolist()
module_4_genes = sm_modules[sm_modules['module'] == 4]['id'].tolist()
module_7_genes = sm_modules[sm_modules['module'] == 7]['id'].tolist()
module_3_genes = sm_modules[sm_modules['module'] == 3]['id'].tolist()


genes_of_interest = top_slope_genes + bot_slope_genes module_3_genes + module_4_genes + module_2_genes + module_1_genes + module_7_genes
genes_of_interest = top_slope_genes + bot_slope_genes

good_genes = []
for gene in genes_of_interest:
    if gene in sm_z_expr_df.columns:
        good_genes.append(gene)
temp = sm_z_expr_df[good_genes].copy()

vcenter = 0
#vmin, vmax = temp.min().min(), temp.max().max()
vmin, vmax = -2, 3
normalize = mcolors.TwoSlopeNorm(vcenter=vcenter, vmin=vmin, vmax=vmax)
colormap = cm.RdYlBu_r

g = sns.clustermap(temp.T, figsize = (10, 15), col_cluster = False, row_cluster = True, cmap=colormap)
plt.subplots_adjust(bottom = 0.3, right = 0.7)
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/mesenchyme_markers_heatmap.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
test